In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 05 · Torneo de modelos

> **Esqueleto.** Guía completa: `curso/kit-estudiante/guia-modelado-y-optimizacion.md`, sección 2.

Objetivo: elegir **uno o dos finalistas** para optimizar, con evidencia y sin abrir la prueba. Reglas:

1. **Mismos datos y mismos pliegues** para todos (por entidad si hay entidades repetidas).
2. Una **línea base** que cualquier modelo serio debe vencer.
3. **Guardar cada pliegue**, no solo el promedio, para comparar de a pares.

El torneo **descarta**, no corona: la optimización puede cambiar el orden y solo la prueba final (clase 4)
dice qué tan bueno es el modelo elegido.

## 0. Cargar los datos de entrenamiento

In [ ]:
import pandas as pd

# === Configuración: completa estas dos líneas (las mismas de tu notebook 03) ===
ID_COL = None     # TODO: columna que identifica a la entidad que se repite (None si no hay entidades repetidas)
TARGET = None     # TODO: tu columna objetivo

train_path = Path.cwd().resolve().parent / "data" / "processed" / "train.joblib"
if not train_path.exists():
    train = None
    print("Todavía no hay datos de entrenamiento: completa y ejecuta antes el notebook 03_preprocesamiento.")
else:
    import joblib
    train = joblib.load(train_path)
    print(f"entrenamiento: {train.shape[0]:,} filas x {train.shape[1]} columnas (la prueba NO se abre hoy)")

listo = train is not None and TARGET is not None
if train is not None and TARGET is None:
    print("Define TARGET (y, si hay entidades repetidas, ID_COL) arriba para continuar.")

## 1. Validación común (los mismos pliegues para todos)

In [ ]:
if not listo:
    pliegues = None
    print("Sin datos o sin TARGET: esta celda crea los pliegues cuando estén.")
else:
    from sklearn.model_selection import GroupKFold, KFold, StratifiedGroupKFold, StratifiedKFold

    X, y = train.drop(columns=[TARGET]), train[TARGET]
    es_regresion = pd.api.types.is_numeric_dtype(y) and y.nunique() > 20   # ajusta si tu objetivo es otro
    if ID_COL is not None:
        cv = GroupKFold(n_splits=5) if es_regresion else StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
        pliegues = list(cv.split(X, y, groups=X[ID_COL]))
        for i_ent, i_val in pliegues:   # comprobación: ninguna entidad en los dos lados de un pliegue
            assert set(X.iloc[i_ent][ID_COL]).isdisjoint(set(X.iloc[i_val][ID_COL])), "hay entidades en los dos lados"
    else:
        cv = KFold(n_splits=5, shuffle=True, random_state=42) if es_regresion else StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        pliegues = list(cv.split(X, y))
    print(f"{len(pliegues)} pliegues, los MISMOS para todo lo que compares hoy.")

## 2. Pipeline y métricas

Reporta **dos familias de métricas**, siempre las dos (guía, sección 2.2):

- **De negocio (la principal):** refleja la decisión del negocio y decide quién gana.
- **Técnicas:** AUC macro y de la clase de interés, PR-AUC, exactitud balanceada, precisión y recall **por clase**,
  F1 macro y log loss (en regresión: RMSE, MAE y R²). No dependen del negocio y muestran lo que la métrica de
  negocio puede esconder, por ejemplo una clase que casi nunca se predice.

No uses la exactitud simple si las clases están desbalanceadas: usa la exactitud balanceada.

In [ ]:
try:
    from proyecto.models.train import build_pipeline   # TODO: crea build_pipeline(modelo, ...) en src/proyecto/models/train.py
except ModuleNotFoundError:
    build_pipeline = None
    print("TODO: crea src/proyecto/models/train.py con build_pipeline(modelo): preprocesamiento + modelo dentro de un Pipeline.")

scoring = None   # TODO: dict con la métrica de NEGOCIO y las técnicas que cross_validate puede calcular,
                 # p. ej. {"costo": make_scorer(tu_costo, greater_is_better=False), "auc": "roc_auc_ovr",
                 #         "exactitud_balanceada": "balanced_accuracy", "f1": "f1_macro"}
                 # (el recall por clase y el AUC de la clase de interés se calculan aparte: ver sección 5.1)
METRICA = None   # TODO: nombre de tu métrica principal dentro de scoring (si es negativa, multiplícala por -1 al leerla)

## 3. Candidatos

Mínimo **3 modelos y una línea base**: algo simple e interpretable, un bosque y un modelo de boosting.

In [ ]:
candidatos = {
    # "línea base": ...,          # TODO: clase más frecuente (clasificación) o la media (regresión)
    # "modelo simple": ...,       # TODO: regresión logística / lineal
    # "bosque aleatorio": ...,    # TODO
    # "boosting": ...,            # TODO: LightGBM o XGBoost
}
print(f"{len(candidatos)} candidatos definidos" if candidatos else "TODO: define tus candidatos (mínimo 3 y una línea base).")

## 4. Correr el torneo (una fila por pliegue)

In [ ]:
por_pliegue = {}   # nombre del modelo -> DataFrame con una fila por pliegue

faltan = [nombre for nombre, valor in {"pliegues": pliegues, "build_pipeline": build_pipeline, "scoring": scoring,
                                          "candidatos": candidatos}.items() if not valor]
if faltan:
    print("Pendiente: completa", ", ".join(faltan), "para correr el torneo.")
else:
    from sklearn.model_selection import cross_validate

    for nombre, modelo in candidatos.items():
        res = cross_validate(build_pipeline(modelo), X, y, cv=pliegues, scoring=scoring, n_jobs=1, return_train_score=False)
        por_pliegue[nombre] = pd.DataFrame({k.replace("test_", ""): v for k, v in res.items() if k.startswith("test_")})
        por_pliegue[nombre]["segundos_de_ajuste"] = res["fit_time"]
        print(f"{nombre:28s} listo")

## 5. Tabla resumen y variabilidad

In [ ]:
import numpy as np

if not por_pliegue or METRICA is None:
    print("Sin resultados todavía: esta celda mostrará la media ± desviación de cada modelo.")
else:
    resumen = pd.DataFrame({
        nombre: {"métrica (media)": f[METRICA].abs().mean(), "± entre pliegues": f[METRICA].abs().std(ddof=1),
                 "segundos de ajuste": f["segundos_de_ajuste"].mean()}
        for nombre, f in por_pliegue.items()
    }).T.sort_values("métrica (media)")   # TODO: invierte el orden si en tu métrica MÁS es mejor
    display(resumen.round(4))

    import matplotlib.pyplot as plt
    orden = list(resumen.index)
    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.boxplot([por_pliegue[n][METRICA].abs() for n in orden], vert=False, tick_labels=orden, widths=0.5)
    ax.invert_yaxis()
    ax.set_xlabel("métrica por pliegue")
    plt.tight_layout()
    plt.show()

### 5.1 Métricas técnicas (no solo la de negocio)

El costo (o tu métrica de negocio) dice qué conviene al **negocio**. Calcula también las **técnicas** de cada
modelo, con la misma validación. Las de *ranking* (AUC, PR-AUC) usan las probabilidades; las de clasificación
(recall, precisión, F1, exactitud balanceada) usan la decisión que tomaría tu negocio.

Función sugerida: `metricas_tecnicas(y_real, proba, decision)` de la guía, secciones 2.2 y 2.3. Mira especialmente
el **recall de cada clase**: una clase que casi nunca se predice es un riesgo aunque el costo se vea bien.

In [ ]:
# TODO: una fila por modelo con las métricas técnicas (media entre pliegues).
# Para cada modelo y pliegue necesitas las probabilidades (proba) y la decisión (decision).
# Columnas esperadas: AUC macro, AUC clase de interés, PR-AUC, exactitud balanceada, F1 macro,
#                     precisión clase de interés, recall de cada clase, log loss.
tecnicas = None   # TODO: DataFrame (una fila por modelo)

if tecnicas is None:
    print("TODO: calcula las métricas técnicas de cada modelo (guía, secciones 2.2 y 2.3) y guárdalas en `tecnicas`.")
else:
    display(tecnicas.round(3))

### 5.2 ¿Coinciden los rankings de negocio y técnico?

Un modelo puede ser el mejor para el negocio y no serlo en una métrica técnica, y al revés. Si dos modelos tienen un
AUC casi igual pero costos distintos, la diferencia viene de **cómo deciden**, no de qué tan bien ordenan.

In [ ]:
if tecnicas is None or "resumen" not in globals():
    print("Cuando tengas `resumen` y `tecnicas`, compara aquí las posiciones de cada modelo en cada métrica.")
else:
    # TODO: tabla de posiciones (1 = mejor) por cada métrica, con .rank(); ¿el mejor por la métrica de negocio
    #       también es el mejor por AUC y por F1 macro?
    # TODO: ¿hay alguna clase cuyo recall sea casi nulo en algún modelo? ¿cuál y por qué?
    pass

**Mis conclusiones sobre las métricas técnicas** (escríbelas con tus palabras):

- ¿El mejor por la métrica de negocio también es el mejor por AUC? TODO
- ¿Qué modelo es más conservador y qué clase sacrifica (recall bajo)? TODO
- ¿Qué implica eso para el negocio? TODO

## 6. Comparación pareada

Como todos se evaluaron en **los mismos pliegues**, puedes comparar pliegue a pliegue: ¿gana siempre el mismo?

In [ ]:
def comparar(mejor, otro):
    """Ambos: valores por pliegue (menor es mejor). 'diferencia' positiva: 'mejor' gana."""
    dif = np.asarray(otro) - np.asarray(mejor)
    ruido = dif.std(ddof=1)
    return {
        "diferencia media": dif.mean(),
        "pliegues que gana": f"{(dif > 0).sum()} de {len(dif)}",
        "diferencia / ruido": dif.mean() / ruido if ruido > 0 else np.inf,
    }


if not por_pliegue or METRICA is None or len(por_pliegue) < 2:
    print("Sin resultados todavía: aquí se comparan de a pares los mejores modelos.")
else:
    mejores = list(resumen.index[:3])
    pares = []
    for i in range(len(mejores)):
        for j in range(i + 1, len(mejores)):
            r = comparar(por_pliegue[mejores[i]][METRICA].abs(), por_pliegue[mejores[j]][METRICA].abs())
            pares.append({"mejor": mejores[i], "contra": mejores[j], **r})
    display(pd.DataFrame(pares).round(4))

## 7. ¿Cuánto vale la diferencia?

Traduce la diferencia de la métrica a **dinero o volumen** de tu caso: `métrica por decisión × decisiones al mes`.

In [ ]:
DECISIONES_AL_MES = None   # TODO: cuántas decisiones/predicciones haría tu startup en un mes (una estimación razonable)

if not por_pliegue or METRICA is None or DECISIONES_AL_MES is None:
    print("Define DECISIONES_AL_MES y corre el torneo para ver el costo mensual de cada modelo.")
else:
    mensual = (resumen["métrica (media)"] * DECISIONES_AL_MES).rename("costo mensual (unidades)")
    display(mensual.to_frame().style.format("{:,.0f}"))

## 8. Finalistas

**1 o 2**, mirando en este orden: consistencia entre pliegues, métrica principal, tiempo de ajuste y
simplicidad, e interpretabilidad si tu caso la exige.

| Finalista | Por qué | Qué podría cambiar con la optimización |
|---|---|---|
| TODO | TODO | TODO |

**Por qué el torneo no elige un campeón:** TODO (explícalo con tus palabras).

## Entregable de esta etapa

- [ ] Al menos **3 modelos y una línea base**, evaluados en **los mismos pliegues**.
- [ ] Una fila **por pliegue** guardada para cada modelo.
- [ ] **Métricas técnicas** de cada modelo (AUC, PR-AUC, exactitud balanceada, precisión y recall por clase) y la comparación entre el ranking de negocio y el técnico.
- [ ] Comparación pareada de los mejores (pliegues que gana y diferencia / ruido).
- [ ] La diferencia traducida a **dinero o volumen**.
- [ ] **1–2 finalistas** justificados en `reports/05_torneo.md`.
- [ ] El código reutilizable está en `src/proyecto/models/` y el notebook solo lo llama.